# Chapter 9.3: Monitoring & Observability

Production LLM serving demands comprehensive observability: metrics, logging, tracing,
and alerting. This notebook builds a complete monitoring stack from scratch.

## Learning Objectives
- Understand vLLM and SGLang Prometheus metrics
- Build Grafana dashboards for LLM serving
- Create custom Prometheus metrics and alert rules
- Implement structured logging and request tracing
- Set up distributed tracing with OpenTelemetry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part9/chapter_9.3_monitoring.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part9/chapter_9.3_monitoring.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import json
import time
import random
import textwrap
from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict

CONFIG_DIR = Path("./monitoring_configs")
CONFIG_DIR.mkdir(exist_ok=True)

def write_config(filename, content, subdir=None):
    target_dir = CONFIG_DIR / subdir if subdir else CONFIG_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / filename
    path.write_text(textwrap.dedent(content).strip() + "\n")
    print(f"Written: {path}")
    print(path.read_text()[:500])
    if len(path.read_text()) > 500:
        print("... (truncated)")
    return path

print("Setup complete.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_monitoring_stack():
    """Monitoring stack: vLLM/SGLang -> Prometheus -> Grafana -> Alertmanager -> PagerDuty/Slack."""
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.set_xlim(0, 20); ax.set_ylim(0, 6); ax.axis('off')
    ax.set_title('LLM Serving Monitoring Stack', fontsize=16, fontweight='bold', pad=15)

    components = [
        (0.5, 2.0, 3.0, 2.0, 'vLLM / SGLang\n(/metrics endpoint)', '#4A90D9'),
        (4.5, 2.0, 3.0, 2.0, 'Prometheus\n(Scrape & Store\nTime Series)', '#E74C3C'),
        (8.5, 2.0, 3.0, 2.0, 'Grafana\n(Dashboards &\nVisualization)', '#27AE60'),
        (12.5, 2.0, 3.0, 2.0, 'Alertmanager\n(Alert Routing\n& Dedup)', '#F39C12'),
        (16.5, 3.2, 3.0, 1.2, 'PagerDuty', '#8E44AD'),
        (16.5, 1.5, 3.0, 1.2, 'Slack', '#8E44AD'),
    ]

    for x, y, w, h, label, color in components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                              facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')

    # Arrows between components
    arrow_pairs = [(3.5, 4.5), (7.5, 8.5), (11.5, 12.5)]
    for x1, x2 in arrow_pairs:
        ax.annotate('', xy=(x2, 3.0), xytext=(x1, 3.0),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

    # Fork to PagerDuty and Slack
    ax.annotate('', xy=(16.5, 3.8), xytext=(15.5, 3.2),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2, connectionstyle='arc3,rad=-0.15'))
    ax.annotate('', xy=(16.5, 2.1), xytext=(15.5, 2.8),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2, connectionstyle='arc3,rad=0.15'))

    # Labels on arrows
    labels = [('scrape\n15s interval', 4.0), ('query &\nvisualize', 8.0), ('fire\nalerts', 12.0)]
    for label, x in labels:
        ax.text(x, 4.3, label, ha='center', va='bottom', fontsize=7, color='#555', style='italic')

    # Metric examples below
    ax.text(2.0, 0.8, 'Metrics:\n- gpu_cache_usage\n- request_latency\n- tokens_per_sec',
            fontsize=7, color='#4A90D9', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', edgecolor='#4A90D9', alpha=0.7))
    ax.text(10.0, 0.8, 'Dashboards:\n- Throughput\n- Latency P50/P99\n- KV Cache',
            fontsize=7, color='#27AE60', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', edgecolor='#27AE60', alpha=0.7))

    plt.tight_layout()
    plt.savefig('monitoring_stack.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_monitoring_stack()

## 1. Demo: Prometheus Metrics Exposed by vLLM and SGLang

Both vLLM and SGLang expose Prometheus-compatible metrics at `/metrics`.
Let us examine and understand the key metrics.

In [ ]:
# Key vLLM metrics and their meanings
VLLM_METRICS = {
    "vllm:num_requests_running": {
        "type": "gauge",
        "description": "Number of requests currently being processed",
        "alert_threshold": "Watch for sustained high values indicating saturation",
    },
    "vllm:num_requests_waiting": {
        "type": "gauge",
        "description": "Number of requests waiting in the queue",
        "alert_threshold": "> 50 sustained for 2 min suggests scaling is needed",
    },
    "vllm:num_requests_swapped": {
        "type": "gauge",
        "description": "Number of requests swapped to CPU memory",
        "alert_threshold": "> 0 suggests GPU memory pressure",
    },
    "vllm:gpu_cache_usage_perc": {
        "type": "gauge",
        "description": "Percentage of GPU KV cache blocks in use",
        "alert_threshold": "> 95% critical, > 85% warning",
    },
    "vllm:cpu_cache_usage_perc": {
        "type": "gauge",
        "description": "Percentage of CPU KV cache blocks in use (swap space)",
        "alert_threshold": "> 50% indicates heavy swapping",
    },
    "vllm:request_success_total": {
        "type": "counter",
        "description": "Total number of successfully completed requests",
        "alert_threshold": "Use rate() for throughput monitoring",
    },
    "vllm:avg_prompt_throughput_toks_per_s": {
        "type": "gauge",
        "description": "Average prompt processing throughput (tokens/s)",
        "alert_threshold": "Drop > 50% from baseline is concerning",
    },
    "vllm:avg_generation_throughput_toks_per_s": {
        "type": "gauge",
        "description": "Average generation throughput (tokens/s)",
        "alert_threshold": "Drop > 50% from baseline is concerning",
    },
    "vllm:time_to_first_token_seconds": {
        "type": "histogram",
        "description": "Time from request receipt to first token generated",
        "alert_threshold": "P99 > 5s for interactive use cases",
    },
    "vllm:time_per_output_token_seconds": {
        "type": "histogram",
        "description": "Inter-token latency during generation",
        "alert_threshold": "P99 > 100ms causes noticeable stutter in streaming",
    },
    "vllm:e2e_request_latency_seconds": {
        "type": "histogram",
        "description": "End-to-end request latency",
        "alert_threshold": "Depends on SLO, often P99 < 30s",
    },
}

print(f"{'Metric':<50} {'Type':<12} {'Alert Threshold'}")
print("=" * 110)
for name, info in VLLM_METRICS.items():
    print(f"{name:<50} {info['type']:<12} {info['alert_threshold']}")

In [ ]:
# Simulate parsing /metrics output

SAMPLE_METRICS_OUTPUT = """\
# HELP vllm:num_requests_running Number of requests currently running
# TYPE vllm:num_requests_running gauge
vllm:num_requests_running 12
# HELP vllm:num_requests_waiting Number of requests waiting
# TYPE vllm:num_requests_waiting gauge
vllm:num_requests_waiting 3
# HELP vllm:gpu_cache_usage_perc GPU KV cache usage percentage
# TYPE vllm:gpu_cache_usage_perc gauge
vllm:gpu_cache_usage_perc 0.72
# HELP vllm:avg_generation_throughput_toks_per_s Average generation throughput
# TYPE vllm:avg_generation_throughput_toks_per_s gauge
vllm:avg_generation_throughput_toks_per_s 1523.4
# HELP vllm:avg_prompt_throughput_toks_per_s Average prompt throughput
# TYPE vllm:avg_prompt_throughput_toks_per_s gauge
vllm:avg_prompt_throughput_toks_per_s 8921.1
# HELP vllm:e2e_request_latency_seconds_bucket End-to-end latency histogram
# TYPE vllm:e2e_request_latency_seconds histogram
vllm:e2e_request_latency_seconds_bucket{le="0.5"} 45
vllm:e2e_request_latency_seconds_bucket{le="1.0"} 120
vllm:e2e_request_latency_seconds_bucket{le="2.5"} 280
vllm:e2e_request_latency_seconds_bucket{le="5.0"} 350
vllm:e2e_request_latency_seconds_bucket{le="10.0"} 395
vllm:e2e_request_latency_seconds_bucket{le="30.0"} 400
vllm:e2e_request_latency_seconds_bucket{le="+Inf"} 400
vllm:e2e_request_latency_seconds_sum 1245.6
vllm:e2e_request_latency_seconds_count 400
"""

def parse_prometheus_metrics(text: str) -> dict:
    """Parse Prometheus text format into a dict."""
    metrics = {}
    for line in text.strip().split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        # Handle labels: metric_name{labels} value
        if "{" in line:
            name_labels, value = line.rsplit(" ", 1)
            name = name_labels.split("{")[0]
            labels = name_labels.split("{")[1].rstrip("}")
            key = f"{name}{{{labels}}}"
        else:
            parts = line.split()
            key = parts[0]
            value = parts[1]
        metrics[key] = float(value)
    return metrics

parsed = parse_prometheus_metrics(SAMPLE_METRICS_OUTPUT)
print("Parsed metrics:")
for k, v in parsed.items():
    print(f"  {k}: {v}")

In [ ]:
# Compute percentiles from histogram buckets

def compute_histogram_percentile(buckets: list[tuple[float, int]], percentile: float) -> float:
    """Approximate a percentile from Prometheus histogram buckets.
    
    Args:
        buckets: list of (le_bound, cumulative_count) sorted by le_bound
        percentile: 0-1, e.g. 0.99 for P99
    """
    total = buckets[-1][1]  # +Inf bucket has total count
    target = total * percentile
    
    prev_bound = 0
    prev_count = 0
    
    for bound, count in buckets:
        if count >= target:
            # Linear interpolation within this bucket
            fraction = (target - prev_count) / max(1, count - prev_count)
            return prev_bound + fraction * (bound - prev_bound)
        prev_bound = bound
        prev_count = count
    
    return buckets[-1][0]

# Example: compute P50, P95, P99 from our sample data
latency_buckets = [
    (0.5, 45),
    (1.0, 120),
    (2.5, 280),
    (5.0, 350),
    (10.0, 395),
    (30.0, 400),
    (float('inf'), 400),
]

for p in [0.50, 0.75, 0.90, 0.95, 0.99]:
    val = compute_histogram_percentile(latency_buckets, p)
    print(f"  P{int(p*100):>2}: {val:.2f}s")

avg = parsed.get("vllm:e2e_request_latency_seconds_sum", 0) / max(1, parsed.get("vllm:e2e_request_latency_seconds_count", 1))
print(f"  Avg: {avg:.2f}s")

## 2. Demo: Grafana Dashboard JSON

A complete Grafana dashboard for monitoring vLLM serving.

In [ ]:
# Build a Grafana dashboard programmatically

def make_panel(title, panel_type, targets, grid_pos, unit="", thresholds=None):
    """Create a Grafana panel definition."""
    panel = {
        "title": title,
        "type": panel_type,
        "gridPos": grid_pos,
        "datasource": {"type": "prometheus", "uid": "prometheus"},
        "targets": [
            {"expr": t["expr"], "legendFormat": t.get("legend", ""), "refId": chr(65+i)}
            for i, t in enumerate(targets)
        ],
        "fieldConfig": {
            "defaults": {
                "unit": unit,
                "thresholds": thresholds or {"mode": "absolute", "steps": []}
            }
        }
    }
    return panel

dashboard = {
    "dashboard": {
        "id": None,
        "uid": "vllm-serving-dashboard",
        "title": "vLLM Serving Dashboard",
        "tags": ["vllm", "llm", "serving"],
        "timezone": "browser",
        "refresh": "10s",
        "time": {"from": "now-1h", "to": "now"},
        "panels": [
            # Row 1: Key Stats
            make_panel(
                "Active Requests", "stat",
                [{"expr": "vllm:num_requests_running", "legend": "Running"}],
                {"h": 4, "w": 4, "x": 0, "y": 0},
                thresholds={"mode": "absolute", "steps": [
                    {"color": "green", "value": None},
                    {"color": "yellow", "value": 50},
                    {"color": "red", "value": 100}
                ]}
            ),
            make_panel(
                "Queued Requests", "stat",
                [{"expr": "vllm:num_requests_waiting", "legend": "Waiting"}],
                {"h": 4, "w": 4, "x": 4, "y": 0},
                thresholds={"mode": "absolute", "steps": [
                    {"color": "green", "value": None},
                    {"color": "yellow", "value": 10},
                    {"color": "red", "value": 50}
                ]}
            ),
            make_panel(
                "GPU KV Cache Usage", "gauge",
                [{"expr": "vllm:gpu_cache_usage_perc", "legend": "GPU Cache"}],
                {"h": 4, "w": 4, "x": 8, "y": 0},
                unit="percentunit",
                thresholds={"mode": "absolute", "steps": [
                    {"color": "green", "value": None},
                    {"color": "yellow", "value": 0.8},
                    {"color": "red", "value": 0.95}
                ]}
            ),
            make_panel(
                "Request Throughput", "stat",
                [{"expr": "rate(vllm:request_success_total[5m])", "legend": "req/s"}],
                {"h": 4, "w": 4, "x": 12, "y": 0},
                unit="reqps"
            ),
            make_panel(
                "Generation Throughput", "stat",
                [{"expr": "vllm:avg_generation_throughput_toks_per_s", "legend": "tok/s"}],
                {"h": 4, "w": 4, "x": 16, "y": 0},
                unit="short"
            ),
            make_panel(
                "Prompt Throughput", "stat",
                [{"expr": "vllm:avg_prompt_throughput_toks_per_s", "legend": "tok/s"}],
                {"h": 4, "w": 4, "x": 20, "y": 0},
                unit="short"
            ),
            # Row 2: Latency charts
            make_panel(
                "E2E Request Latency", "timeseries",
                [
                    {"expr": "histogram_quantile(0.50, rate(vllm:e2e_request_latency_seconds_bucket[5m]))", "legend": "P50"},
                    {"expr": "histogram_quantile(0.95, rate(vllm:e2e_request_latency_seconds_bucket[5m]))", "legend": "P95"},
                    {"expr": "histogram_quantile(0.99, rate(vllm:e2e_request_latency_seconds_bucket[5m]))", "legend": "P99"},
                ],
                {"h": 8, "w": 12, "x": 0, "y": 4},
                unit="s"
            ),
            make_panel(
                "Time to First Token", "timeseries",
                [
                    {"expr": "histogram_quantile(0.50, rate(vllm:time_to_first_token_seconds_bucket[5m]))", "legend": "P50"},
                    {"expr": "histogram_quantile(0.95, rate(vllm:time_to_first_token_seconds_bucket[5m]))", "legend": "P95"},
                    {"expr": "histogram_quantile(0.99, rate(vllm:time_to_first_token_seconds_bucket[5m]))", "legend": "P99"},
                ],
                {"h": 8, "w": 12, "x": 12, "y": 4},
                unit="s"
            ),
            # Row 3: Cache and throughput
            make_panel(
                "KV Cache Usage Over Time", "timeseries",
                [
                    {"expr": "vllm:gpu_cache_usage_perc", "legend": "GPU Cache"},
                    {"expr": "vllm:cpu_cache_usage_perc", "legend": "CPU Cache"},
                ],
                {"h": 8, "w": 12, "x": 0, "y": 12},
                unit="percentunit"
            ),
            make_panel(
                "Token Throughput Over Time", "timeseries",
                [
                    {"expr": "vllm:avg_prompt_throughput_toks_per_s", "legend": "Prompt tok/s"},
                    {"expr": "vllm:avg_generation_throughput_toks_per_s", "legend": "Gen tok/s"},
                ],
                {"h": 8, "w": 12, "x": 12, "y": 12},
                unit="short"
            ),
        ],
    },
    "overwrite": True
}

# Write dashboard JSON
dashboard_path = CONFIG_DIR / "grafana" / "vllm-dashboard.json"
dashboard_path.parent.mkdir(parents=True, exist_ok=True)
dashboard_path.write_text(json.dumps(dashboard, indent=2))
print(f"Dashboard written to: {dashboard_path}")
print(f"Panels: {len(dashboard['dashboard']['panels'])}")
for p in dashboard['dashboard']['panels']:
    print(f"  - {p['title']} ({p['type']})")

## 3. Demo: Custom Metrics — Request Latency Histogram and Token Throughput

In [ ]:
# Custom metrics implementation using prometheus_client patterns

import math

class Histogram:
    """Simple histogram for latency tracking (mirrors prometheus_client.Histogram)."""
    
    DEFAULT_BUCKETS = (0.005, 0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5,
                       0.75, 1.0, 2.5, 5.0, 7.5, 10.0, 30.0, float('inf'))
    
    def __init__(self, name: str, description: str, buckets=None):
        self.name = name
        self.description = description
        self.buckets = buckets or self.DEFAULT_BUCKETS
        self._counts = {b: 0 for b in self.buckets}
        self._sum = 0.0
        self._count = 0
    
    def observe(self, value: float):
        """Record an observation."""
        self._sum += value
        self._count += 1
        for b in self.buckets:
            if value <= b:
                self._counts[b] += 1
    
    def percentile(self, p: float) -> float:
        """Compute approximate percentile."""
        target = self._count * p
        cumulative = 0
        prev_bound = 0
        for b in sorted(self.buckets):
            cumulative += self._counts[b] - (cumulative)  # bucket count
            # Actually, _counts stores cumulative already in observe
            pass
        # Simplified: use sorted buckets with cumulative counts
        sorted_buckets = sorted(self.buckets)
        prev_bound = 0
        prev_cum = 0
        for b in sorted_buckets:
            cum = self._counts[b]
            if cum >= target and b != float('inf'):
                fraction = (target - prev_cum) / max(1, cum - prev_cum)
                return prev_bound + fraction * (b - prev_bound)
            prev_bound = b
            prev_cum = cum
        return sorted_buckets[-2] if len(sorted_buckets) > 1 else 0
    
    def to_prometheus(self) -> str:
        """Export in Prometheus text format."""
        lines = [
            f"# HELP {self.name} {self.description}",
            f"# TYPE {self.name} histogram",
        ]
        for b in sorted(self.buckets):
            le = "+Inf" if b == float('inf') else str(b)
            lines.append(f'{self.name}_bucket{{le="{le}"}} {self._counts[b]}')
        lines.append(f"{self.name}_sum {self._sum}")
        lines.append(f"{self.name}_count {self._count}")
        return "\n".join(lines)


class Counter:
    """Simple counter metric."""
    def __init__(self, name, description):
        self.name = name
        self.description = description
        self._value = 0
    
    def inc(self, amount=1):
        self._value += amount
    
    def to_prometheus(self):
        return (f"# HELP {self.name} {self.description}\n"
                f"# TYPE {self.name} counter\n"
                f"{self.name} {self._value}")


class Gauge:
    """Simple gauge metric."""
    def __init__(self, name, description):
        self.name = name
        self.description = description
        self._value = 0.0
    
    def set(self, value):
        self._value = value
    
    def inc(self, amount=1):
        self._value += amount
    
    def dec(self, amount=1):
        self._value -= amount
    
    def to_prometheus(self):
        return (f"# HELP {self.name} {self.description}\n"
                f"# TYPE {self.name} gauge\n"
                f"{self.name} {self._value}")


# Create custom metrics for an LLM serving system
request_latency = Histogram(
    "llm_request_latency_seconds",
    "End-to-end request latency in seconds",
    buckets=(0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 30.0, 60.0, float('inf'))
)
ttft = Histogram(
    "llm_time_to_first_token_seconds",
    "Time to first token in seconds",
    buckets=(0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0, float('inf'))
)
tokens_generated = Counter("llm_tokens_generated_total", "Total tokens generated")
active_requests = Gauge("llm_active_requests", "Currently active requests")

# Simulate some traffic
rng = random.Random(42)
for _ in range(1000):
    lat = rng.expovariate(1/3.0)  # Exponential with mean 3s
    request_latency.observe(lat)
    ttft.observe(lat * rng.uniform(0.05, 0.2))  # TTFT is fraction of total
    tokens_generated.inc(rng.randint(20, 500))

active_requests.set(12)

print("Latency percentiles:")
for p in [0.50, 0.90, 0.95, 0.99]:
    print(f"  P{int(p*100)}: {request_latency.percentile(p):.3f}s")
print(f"  Avg: {request_latency._sum / request_latency._count:.3f}s")
print(f"\nTotal tokens: {tokens_generated._value}")
print(f"Active requests: {active_requests._value}")

In [ ]:
# Export all metrics in Prometheus format
print("=== Prometheus Metrics Output ===")
for metric in [request_latency, ttft, tokens_generated, active_requests]:
    print(metric.to_prometheus())
    print()

## 4. Alert Rules for SLO Violations

In [ ]:
# Comprehensive alert rules for LLM serving
alert_rules = """\
groups:
  - name: llm_serving_slos
    interval: 30s
    rules:
      # === Latency SLOs ===
      - alert: HighP99Latency
        expr: |
          histogram_quantile(0.99,
            rate(vllm:e2e_request_latency_seconds_bucket[5m])
          ) > 30
        for: 5m
        labels:
          severity: warning
          team: ml-platform
        annotations:
          summary: "P99 request latency exceeds 30s SLO"
          description: "P99 latency is {{ $value | humanizeDuration }}"
          runbook: "https://wiki.example.com/runbooks/llm-high-latency"

      - alert: HighTTFT
        expr: |
          histogram_quantile(0.95,
            rate(vllm:time_to_first_token_seconds_bucket[5m])
          ) > 5
        for: 3m
        labels:
          severity: warning
        annotations:
          summary: "P95 Time-to-First-Token exceeds 5s"

      # === Capacity ===
      - alert: KVCacheCritical
        expr: vllm:gpu_cache_usage_perc > 0.95
        for: 2m
        labels:
          severity: critical
        annotations:
          summary: "GPU KV cache usage above 95%"
          description: "Cache at {{ $value | humanizePercentage }}. New requests may be rejected."

      - alert: HighQueueDepth
        expr: vllm:num_requests_waiting > 50
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "Request queue depth above 50 for 5 minutes"
          description: "{{ $value }} requests waiting. Consider scaling up."

      - alert: RequestSwapping
        expr: vllm:num_requests_swapped > 0
        for: 1m
        labels:
          severity: warning
        annotations:
          summary: "Requests being swapped to CPU (GPU memory pressure)"

      # === Availability ===
      - alert: ServiceDown
        expr: up{job="vllm"} == 0
        for: 30s
        labels:
          severity: critical
        annotations:
          summary: "vLLM instance is unreachable"

      - alert: HighErrorRate
        expr: |
          rate(vllm:request_failure_total[5m])
          / (rate(vllm:request_success_total[5m]) + rate(vllm:request_failure_total[5m]))
          > 0.05
        for: 3m
        labels:
          severity: critical
        annotations:
          summary: "Error rate above 5%"

      # === Throughput ===
      - alert: LowThroughput
        expr: vllm:avg_generation_throughput_toks_per_s < 100
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "Generation throughput below 100 tok/s"
"""

write_config("alert_rules.yml", alert_rules, subdir="prometheus")

## 5. Logging Best Practices: Structured Logging and Request Tracing

In [ ]:
import uuid
import logging

class StructuredLogger:
    """Structured JSON logger for LLM serving."""
    
    def __init__(self, service_name: str = "vllm-server"):
        self.service_name = service_name
        self.logger = logging.getLogger(service_name)
        self.logger.setLevel(logging.DEBUG)
        
        # JSON formatter
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter('%(message)s'))
        self.logger.handlers = [handler]
    
    def _log(self, level: str, event: str, **kwargs):
        entry = {
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S.000Z", time.gmtime()),
            "level": level,
            "service": self.service_name,
            "event": event,
            **kwargs
        }
        self.logger.log(
            getattr(logging, level.upper()),
            json.dumps(entry, default=str)
        )
    
    def request_start(self, request_id: str, model: str, 
                      input_tokens: int, params: dict):
        self._log("info", "request_start",
                  request_id=request_id,
                  model=model,
                  input_tokens=input_tokens,
                  max_tokens=params.get("max_tokens"),
                  temperature=params.get("temperature"),
                  stream=params.get("stream", False))
    
    def request_complete(self, request_id: str, output_tokens: int,
                         latency_ms: float, ttft_ms: float):
        self._log("info", "request_complete",
                  request_id=request_id,
                  output_tokens=output_tokens,
                  latency_ms=round(latency_ms, 2),
                  ttft_ms=round(ttft_ms, 2),
                  tokens_per_second=round(output_tokens / (latency_ms / 1000), 1)
                  if latency_ms > 0 else 0)
    
    def request_error(self, request_id: str, error: str, error_type: str):
        self._log("error", "request_error",
                  request_id=request_id,
                  error=error,
                  error_type=error_type)
    
    def system_metric(self, gpu_util: float, kv_cache: float,
                      active_requests: int, queue_depth: int):
        self._log("info", "system_metrics",
                  gpu_utilization=gpu_util,
                  kv_cache_usage=kv_cache,
                  active_requests=active_requests,
                  queue_depth=queue_depth)


# Demo: simulate request lifecycle logging
logger = StructuredLogger("vllm-server")

request_id = str(uuid.uuid4())
print("=== Structured Log Output ===")
logger.request_start(request_id, "llama-3.1-8b", 512, 
                     {"max_tokens": 256, "temperature": 0.7, "stream": True})
logger.request_complete(request_id, 189, 2340.5, 180.3)

print()
err_id = str(uuid.uuid4())
logger.request_error(err_id, "CUDA out of memory", "OOMError")

print()
logger.system_metric(0.82, 0.71, 15, 3)

## 6. Demo: Distributed Tracing with OpenTelemetry

In [ ]:
# OpenTelemetry tracing simulation
# In production, use opentelemetry-api and opentelemetry-sdk

@dataclass
class Span:
    """Simulated OpenTelemetry span."""
    name: str
    trace_id: str
    span_id: str
    parent_id: str = None
    start_time: float = 0.0
    end_time: float = 0.0
    attributes: dict = field(default_factory=dict)
    status: str = "OK"
    
    @property
    def duration_ms(self):
        return (self.end_time - self.start_time) * 1000


class SimpleTracer:
    """Simulated distributed tracer for LLM serving."""
    
    def __init__(self, service_name: str):
        self.service_name = service_name
        self.spans: list[Span] = []
    
    def start_span(self, name: str, trace_id: str = None, 
                   parent_id: str = None, **attributes) -> Span:
        span = Span(
            name=name,
            trace_id=trace_id or uuid.uuid4().hex[:32],
            span_id=uuid.uuid4().hex[:16],
            parent_id=parent_id,
            start_time=time.time(),
            attributes={"service.name": self.service_name, **attributes}
        )
        return span
    
    def end_span(self, span: Span, status: str = "OK", **extra_attrs):
        span.end_time = time.time()
        span.status = status
        span.attributes.update(extra_attrs)
        self.spans.append(span)
    
    def print_trace(self, trace_id: str):
        """Print a trace tree."""
        trace_spans = [s for s in self.spans if s.trace_id == trace_id]
        trace_spans.sort(key=lambda s: s.start_time)
        
        print(f"\nTrace: {trace_id}")
        print(f"{'='*70}")
        
        # Build tree
        for span in trace_spans:
            depth = 0
            parent = span.parent_id
            while parent:
                depth += 1
                parent_span = next((s for s in trace_spans if s.span_id == parent), None)
                parent = parent_span.parent_id if parent_span else None
            
            indent = "  " * depth
            status_icon = "+" if span.status == "OK" else "X"
            print(f"{indent}[{status_icon}] {span.name} ({span.duration_ms:.1f}ms)")
            for k, v in span.attributes.items():
                if k != "service.name":
                    print(f"{indent}    {k}: {v}")


# Simulate a traced LLM request
tracer = SimpleTracer("llm-gateway")
trace_id = uuid.uuid4().hex[:32]

# Root span: HTTP request
root = tracer.start_span("POST /v1/chat/completions", trace_id=trace_id,
                          http_method="POST", http_url="/v1/chat/completions")

# Child: Load balancer routing
lb_span = tracer.start_span("lb.select_backend", trace_id=trace_id,
                             parent_id=root.span_id,
                             lb_strategy="kv_cache_aware")
time.sleep(0.001)
tracer.end_span(lb_span, selected_backend="gpu-2")

# Child: Tokenization
tok_span = tracer.start_span("tokenize", trace_id=trace_id,
                              parent_id=root.span_id,
                              input_length=512)
time.sleep(0.002)
tracer.end_span(tok_span, token_count=128)

# Child: Model inference
infer_span = tracer.start_span("model.generate", trace_id=trace_id,
                                parent_id=root.span_id,
                                model="llama-3.1-8b", batch_position=3)

# Grandchild: Prefill
prefill = tracer.start_span("prefill", trace_id=trace_id,
                             parent_id=infer_span.span_id,
                             prompt_tokens=128)
time.sleep(0.05)
tracer.end_span(prefill)

# Grandchild: Decode
decode = tracer.start_span("decode", trace_id=trace_id,
                            parent_id=infer_span.span_id,
                            max_tokens=256)
time.sleep(0.1)
tracer.end_span(decode, generated_tokens=189)

time.sleep(0.01)
tracer.end_span(infer_span, total_tokens=317)

# End root
tracer.end_span(root, http_status=200, response_tokens=189)

tracer.print_trace(trace_id)

In [ ]:
# OpenTelemetry Collector configuration for LLM serving
otel_config = """\
# OpenTelemetry Collector config for LLM serving
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317
      http:
        endpoint: 0.0.0.0:4318

processors:
  batch:
    timeout: 5s
    send_batch_size: 1024
  
  attributes:
    actions:
      - key: environment
        value: production
        action: upsert
      - key: service.version
        from_attribute: app.version
        action: upsert

exporters:
  jaeger:
    endpoint: jaeger:14250
    tls:
      insecure: true
  
  prometheus:
    endpoint: 0.0.0.0:8889
    namespace: llm

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [batch, attributes]
      exporters: [jaeger]
    metrics:
      receivers: [otlp]
      processors: [batch]
      exporters: [prometheus]
"""

write_config("otel-collector.yaml", otel_config, subdir="otel")

---
## Hands-on Assignments

### Assignment 1: Build a Grafana Dashboard from Scratch

Create a dashboard JSON that monitors both vLLM and SGLang services.

In [ ]:
# ASSIGNMENT 1: Build a comprehensive Grafana dashboard

def create_dashboard(title: str, panels_config: list[dict]) -> dict:
    """Create a Grafana dashboard JSON.
    
    Each panel_config should have:
    - title: str
    - type: "stat" | "gauge" | "timeseries" | "heatmap"
    - queries: list of PromQL expressions
    - unit: str (e.g., "s", "percentunit", "short")
    - row: int (0-indexed, used for grid placement)
    - col: int (0-indexed, used for grid placement)
    - width: int (1-24)
    - height: int (1+)
    
    TODO: Create a dashboard with at least 8 panels covering:
    1. Active requests (stat)
    2. Queue depth (stat)
    3. GPU KV cache usage (gauge)
    4. Request rate (stat)
    5. E2E latency percentiles (timeseries)
    6. TTFT percentiles (timeseries)
    7. Token throughput over time (timeseries)
    8. Error rate (timeseries)
    
    Returns:
        dict: Complete Grafana dashboard JSON
    """
    # YOUR CODE HERE
    panels = []
    for i, config in enumerate(panels_config):
        panel = {
            "id": i + 1,
            "title": config["title"],
            "type": config.get("type", "timeseries"),
            "gridPos": {
                "h": config.get("height", 6),
                "w": config.get("width", 12),
                "x": config.get("col", 0) * 6,
                "y": config.get("row", 0) * 6,
            },
            "targets": [
                {"expr": q, "refId": chr(65 + j)}
                for j, q in enumerate(config.get("queries", []))
            ],
            "fieldConfig": {
                "defaults": {"unit": config.get("unit", "")}
            }
        }
        panels.append(panel)
    
    return {
        "dashboard": {
            "title": title,
            "uid": title.lower().replace(" ", "-"),
            "panels": panels,
            "refresh": "10s",
            "time": {"from": "now-1h", "to": "now"},
        }
    }


# TODO: Define your panels and create the dashboard
my_panels = [
    # Add your panel definitions here
    # Example:
    # {"title": "Active Requests", "type": "stat", 
    #  "queries": ["vllm:num_requests_running"], "row": 0, "col": 0, "width": 4, "height": 4},
]

dashboard = create_dashboard("My LLM Dashboard", my_panels)
print(f"Dashboard created with {len(dashboard['dashboard']['panels'])} panels")
print("Add at least 8 panels to complete the assignment!")

In [ ]:
# ASSIGNMENT 1 - SOLUTION

solution_panels = [
    {"title": "Active Requests", "type": "stat",
     "queries": ["vllm:num_requests_running"],
     "row": 0, "col": 0, "width": 4, "height": 4},
    {"title": "Queue Depth", "type": "stat",
     "queries": ["vllm:num_requests_waiting"],
     "row": 0, "col": 1, "width": 4, "height": 4},
    {"title": "GPU KV Cache", "type": "gauge",
     "queries": ["vllm:gpu_cache_usage_perc"],
     "unit": "percentunit",
     "row": 0, "col": 2, "width": 4, "height": 4},
    {"title": "Request Rate", "type": "stat",
     "queries": ["rate(vllm:request_success_total[5m])"],
     "unit": "reqps",
     "row": 0, "col": 3, "width": 4, "height": 4},
    {"title": "E2E Latency Percentiles", "type": "timeseries",
     "queries": [
         "histogram_quantile(0.50, rate(vllm:e2e_request_latency_seconds_bucket[5m]))",
         "histogram_quantile(0.95, rate(vllm:e2e_request_latency_seconds_bucket[5m]))",
         "histogram_quantile(0.99, rate(vllm:e2e_request_latency_seconds_bucket[5m]))",
     ],
     "unit": "s",
     "row": 1, "col": 0, "width": 12, "height": 8},
    {"title": "Time to First Token", "type": "timeseries",
     "queries": [
         "histogram_quantile(0.50, rate(vllm:time_to_first_token_seconds_bucket[5m]))",
         "histogram_quantile(0.99, rate(vllm:time_to_first_token_seconds_bucket[5m]))",
     ],
     "unit": "s",
     "row": 1, "col": 2, "width": 12, "height": 8},
    {"title": "Token Throughput", "type": "timeseries",
     "queries": [
         "vllm:avg_prompt_throughput_toks_per_s",
         "vllm:avg_generation_throughput_toks_per_s",
     ],
     "unit": "short",
     "row": 2, "col": 0, "width": 12, "height": 8},
    {"title": "Error Rate", "type": "timeseries",
     "queries": [
         "rate(vllm:request_failure_total[5m]) / (rate(vllm:request_success_total[5m]) + rate(vllm:request_failure_total[5m]))",
     ],
     "unit": "percentunit",
     "row": 2, "col": 2, "width": 12, "height": 8},
]

solution_dashboard = create_dashboard("LLM Serving Dashboard", solution_panels)
print(f"Solution dashboard: {len(solution_dashboard['dashboard']['panels'])} panels")
for p in solution_dashboard['dashboard']['panels']:
    print(f"  [{p['type']:<12}] {p['title']}")

# Save it
path = CONFIG_DIR / "grafana" / "solution-dashboard.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(solution_dashboard, indent=2))
print(f"\nSaved to: {path}")

### Assignment 2: Create Alert Rules for P99 Latency

In [ ]:
# ASSIGNMENT 2: Design a multi-level alerting system

def generate_latency_alerts(
    metric_name: str = "vllm:e2e_request_latency_seconds",
    warning_p99_threshold: float = 10.0,
    critical_p99_threshold: float = 30.0,
    warning_p50_threshold: float = 3.0,
    evaluation_window: str = "5m",
    for_duration: str = "3m",
) -> str:
    """Generate Prometheus alert rules for latency SLOs.
    
    TODO: Create alert rules for:
    1. P99 latency warning (> warning_p99_threshold for for_duration)
    2. P99 latency critical (> critical_p99_threshold for for_duration)
    3. P50 latency warning (median too high)
    4. Latency spike detection (P99 jumped >3x in 5 minutes)
    5. Multi-window SLO burn rate (error budget consumption)
    
    Returns:
        Valid Prometheus alert rules YAML.
    """
    # YOUR CODE HERE
    rules = f"""\
groups:
  - name: latency_slos
    rules:
      # TODO: Add your alert rules here
      # Rule 1: P99 warning
      # Rule 2: P99 critical  
      # Rule 3: P50 warning
      # Rule 4: Spike detection
      # Rule 5: SLO burn rate
"""
    return rules

print(generate_latency_alerts())

In [ ]:
# ASSIGNMENT 2 - SOLUTION

def generate_latency_alerts_solution(
    metric_name="vllm:e2e_request_latency_seconds",
    warning_p99_threshold=10.0,
    critical_p99_threshold=30.0,
    warning_p50_threshold=3.0,
    evaluation_window="5m",
    for_duration="3m",
):
    return f"""\
groups:
  - name: latency_slos
    interval: 30s
    rules:
      - alert: P99LatencyWarning
        expr: |
          histogram_quantile(0.99,
            rate({metric_name}_bucket[{evaluation_window}])
          ) > {warning_p99_threshold}
        for: {for_duration}
        labels:
          severity: warning
        annotations:
          summary: "P99 latency above {warning_p99_threshold}s"
          description: "P99 latency is {{{{ $value | humanizeDuration }}}}."

      - alert: P99LatencyCritical
        expr: |
          histogram_quantile(0.99,
            rate({metric_name}_bucket[{evaluation_window}])
          ) > {critical_p99_threshold}
        for: {for_duration}
        labels:
          severity: critical
        annotations:
          summary: "P99 latency above {critical_p99_threshold}s (CRITICAL)"

      - alert: P50LatencyWarning
        expr: |
          histogram_quantile(0.50,
            rate({metric_name}_bucket[{evaluation_window}])
          ) > {warning_p50_threshold}
        for: {for_duration}
        labels:
          severity: warning
        annotations:
          summary: "Median latency above {warning_p50_threshold}s"

      - alert: LatencySpike
        expr: |
          histogram_quantile(0.99, rate({metric_name}_bucket[5m]))
          / histogram_quantile(0.99, rate({metric_name}_bucket[30m]))
          > 3
        for: 2m
        labels:
          severity: warning
        annotations:
          summary: "P99 latency spiked 3x vs 30-minute baseline"

      - alert: SLOBurnRateHigh
        expr: |
          (
            rate({metric_name}_bucket{{le="10.0"}}[1h])
            / rate({metric_name}_count[1h])
          ) < 0.99
        for: 5m
        labels:
          severity: critical
        annotations:
          summary: "SLO burn rate too high - less than 99% requests under 10s"
"""

result = generate_latency_alerts_solution()
print(result)
write_config("latency_alerts.yml", result, subdir="prometheus")

### Assignment 3: Implement Custom Prometheus Metrics

In [ ]:
# ASSIGNMENT 3: Build a metrics collector for LLM serving

class LLMMetricsCollector:
    """Collect and export custom metrics for LLM serving.
    
    TODO: Implement the following metrics:
    1. request_latency_seconds (histogram) - E2E request latency
    2. time_to_first_token_seconds (histogram) - TTFT
    3. tokens_per_second (histogram) - Generation speed per request
    4. input_tokens_total (counter) - Total input tokens processed
    5. output_tokens_total (counter) - Total output tokens generated
    6. active_requests (gauge) - Currently processing
    7. cache_hit_ratio (gauge) - Prefix cache hit ratio
    8. request_errors_total (counter, by error_type) - Error counts
    """
    
    def __init__(self):
        # TODO: Initialize all metrics
        self.request_latency = Histogram(
            "llm_request_latency_seconds", "E2E request latency",
            buckets=(0.1, 0.5, 1, 2.5, 5, 10, 30, 60, float('inf'))
        )
        # Add more metrics here...
        pass
    
    def record_request(self, latency_s: float, ttft_s: float,
                       input_tokens: int, output_tokens: int,
                       cache_hit: bool = False):
        """Record metrics for a completed request."""
        # TODO: Update all relevant metrics
        pass
    
    def record_error(self, error_type: str):
        """Record a request error."""
        # TODO: Increment error counter
        pass
    
    def export_prometheus(self) -> str:
        """Export all metrics in Prometheus text format."""
        # TODO: Concatenate all metric exports
        return ""
    
    def summary(self) -> dict:
        """Return a summary of current metrics."""
        # TODO: Return key statistics
        return {}


# Test harness
collector = LLMMetricsCollector()

# Simulate 100 requests
rng = random.Random(42)
for _ in range(100):
    lat = rng.expovariate(1/2.0)
    ttft = lat * rng.uniform(0.05, 0.15)
    inp = rng.randint(50, 2000)
    out = rng.randint(20, 500)
    collector.record_request(lat, ttft, inp, out, cache_hit=rng.random() < 0.3)

# Simulate some errors
for _ in range(5):
    collector.record_error("timeout")
for _ in range(2):
    collector.record_error("oom")

print("Metrics summary:")
print(json.dumps(collector.summary(), indent=2))

In [ ]:
# ASSIGNMENT 3 - SOLUTION

class LLMMetricsCollectorSolution:
    def __init__(self):
        self.request_latency = Histogram(
            "llm_request_latency_seconds", "E2E request latency",
            buckets=(0.1, 0.5, 1, 2.5, 5, 10, 30, 60, float('inf'))
        )
        self.ttft = Histogram(
            "llm_ttft_seconds", "Time to first token",
            buckets=(0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0, float('inf'))
        )
        self.tps = Histogram(
            "llm_tokens_per_second", "Generation speed",
            buckets=(10, 25, 50, 75, 100, 150, 200, 500, float('inf'))
        )
        self.input_tokens = Counter("llm_input_tokens_total", "Total input tokens")
        self.output_tokens = Counter("llm_output_tokens_total", "Total output tokens")
        self.active = Gauge("llm_active_requests", "Active requests")
        self.cache_hits = 0
        self.cache_total = 0
        self.cache_ratio = Gauge("llm_cache_hit_ratio", "Prefix cache hit ratio")
        self.errors = defaultdict(int)
        self.total_requests = 0
    
    def record_request(self, latency_s, ttft_s, input_tokens, output_tokens, cache_hit=False):
        self.request_latency.observe(latency_s)
        self.ttft.observe(ttft_s)
        if latency_s > 0:
            self.tps.observe(output_tokens / latency_s)
        self.input_tokens.inc(input_tokens)
        self.output_tokens.inc(output_tokens)
        self.cache_total += 1
        if cache_hit:
            self.cache_hits += 1
        self.cache_ratio.set(self.cache_hits / max(1, self.cache_total))
        self.total_requests += 1
    
    def record_error(self, error_type):
        self.errors[error_type] += 1
    
    def export_prometheus(self):
        parts = [
            self.request_latency.to_prometheus(),
            self.ttft.to_prometheus(),
            self.tps.to_prometheus(),
            self.input_tokens.to_prometheus(),
            self.output_tokens.to_prometheus(),
            self.active.to_prometheus(),
            self.cache_ratio.to_prometheus(),
        ]
        # Add error counters
        parts.append("# HELP llm_errors_total Request errors by type")
        parts.append("# TYPE llm_errors_total counter")
        for err_type, count in self.errors.items():
            parts.append(f'llm_errors_total{{type="{err_type}"}} {count}')
        return "\n\n".join(parts)
    
    def summary(self):
        return {
            "total_requests": self.total_requests,
            "latency_p50": round(self.request_latency.percentile(0.5), 3),
            "latency_p99": round(self.request_latency.percentile(0.99), 3),
            "ttft_p50": round(self.ttft.percentile(0.5), 3),
            "ttft_p99": round(self.ttft.percentile(0.99), 3),
            "total_input_tokens": self.input_tokens._value,
            "total_output_tokens": self.output_tokens._value,
            "cache_hit_ratio": round(self.cache_hits / max(1, self.cache_total), 3),
            "errors": dict(self.errors),
        }

# Test
collector = LLMMetricsCollectorSolution()
rng = random.Random(42)
for _ in range(100):
    lat = rng.expovariate(1/2.0)
    collector.record_request(lat, lat*rng.uniform(0.05,0.15),
                             rng.randint(50,2000), rng.randint(20,500),
                             cache_hit=rng.random()<0.3)
for _ in range(5): collector.record_error("timeout")
for _ in range(2): collector.record_error("oom")

print("Summary:")
print(json.dumps(collector.summary(), indent=2))
print(f"\nPrometheus export ({len(collector.export_prometheus())} chars)")
print(collector.export_prometheus()[:600])
print("...")

## Summary

This notebook covered:

1. **vLLM/SGLang metrics**: Understanding key Prometheus metrics for LLM serving
2. **Grafana dashboards**: Building dashboards programmatically with PromQL queries
3. **Custom metrics**: Implementing histograms, counters, and gauges for LLM-specific needs
4. **Alert rules**: Multi-level alerting with latency SLOs and capacity warnings
5. **Structured logging**: JSON logging with request tracing
6. **Distributed tracing**: OpenTelemetry spans for end-to-end request visibility

Key metrics to always monitor: GPU KV cache usage, P99 latency, TTFT, queue depth, and error rate.